## 第二章 处理文本数据


- 将文本分割为独立的单词词元和子词词元，然后编码为 LLM 所使用的向量表示


### 2.1 理解词嵌入


- 深度神经网络无法直接处理原始文本，因为文本数据是离散的，无法直接执行数学运算。
- 将数据转化为向量格式的过程称为嵌入 embedding，不同的数据类型如文本，图像，音视频等需要使用不同的嵌入模型。
- embedding 的本质是将离散对象映射到连续向量空间中的点，从而转化为神经网络可以处理的格式。
- RAG，retrieval-augmented generation，是将句子，段落乃至整个文档嵌入的技术。
- 有多种算法和框架来生成词嵌入，早期流行 word2vec。
- 词嵌入的维度 dimension 可以从一维到数千维不等，更高的维度有助于捕获到更细微的关系，同时牺牲计算效率。
- 大语言模型通常会自行生成嵌入，这些嵌入是输入层的一部分，并且在训练中会进行更新。
- 最小的 GPT-2 模型参数为 1.17 亿，嵌入维度为 768，GPT-3 参数为 1750 亿，嵌入维度为 12288。


### 2.2 文本分词


- 将输入文本分割为独立的词元，是生成嵌入向量所必须的预处理步骤。
- 词元即可以是单个的单词，也可以是诸如标点符号之类的特殊字符。


- 采用 Edith Wharton 的短篇小说 The Verdict 为分词文本作为例子。


In [1]:
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x1778b6d44a0>)

In [1]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of characters in the text:", len(raw_text))
print(raw_text[:99])

Total number of characters in the text: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- 将这篇 20479 个字符的短片小说分割为独立的单词和特殊字符，以便于在后续步骤中将其转化为嵌入向量，进而用于 LLM 训练。
- 为了获取词元列表，就需要正确的分割文本。
- 何为正确？如移除空格可以减少运算负担，但是会丢失掉空格在上下文中的作用。


- 下面使用正则来示例如何分割


In [2]:
import re
text = "Hello, world. This,  is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', '', ' ', 'is', ' ', 'a', ' ', 'test.']


将单词与标点符号分离


In [ ]:
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


空白字符仍然存在，用下面的方式分离


In [4]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [5]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


输入了 Hello, world. Is this-- a test?  
输出了'Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?'


In [6]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


In [7]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


### 2.3 将词元转换为词元 ID


- 词元被分割后，还需要生成词元 ID，即 token ID，与之一一映射；
- 为了完成映射，需要构建一张词汇表；
- 首先将训练集中的全部文本分割成独立的词元，
- 然后将这些词元按照字母顺序排列并删除重复词元；
- 然后将唯一的词元聚合到一个词汇表中。


In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [9]:
vocab = {token: integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


- 上面演示了对训练文本分割——去重——构建词汇表的过程；
- 现实中还需要构建一个逆向词汇表，从而将词元 ID 映射回对应的文本；
- 下面演示一个完整的分词器，包括了：
- encode, 通过词汇表将文本映射到整数，以生成词元 ID，
- decode，从整数到字符串的反向映射，将词元 ID 还原回文本，


In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab  # 将词汇表作为类存储，以便于在encoder和decoder方法中访问
        # 创建逆向词汇表，将词元ID映射回原始文本词元
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):  # 将输入文本转换为词元ID列表
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):  # 将词元ID列表转换回原始文本
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)  # 移除标点符号前的空格
        return text

创建一个 SimpleTokenizerV1 类的实例对象，将其应用于 The Verdict 的一段文本中，测试效果


In [11]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
        Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [12]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


- 分词器仅能用于训练集中的文本里，超出范围就会报错；
- 如下所示，Hello 没有在小说 The Verdict 中出现；


In [13]:
text = "Hello, do you like tea?"
print(tokenizer.encode(text))

KeyError: 'Hello'

### 